In [209]:
import numpy as np
import pandas as pd

In [210]:
df_deliveries = pd.read_csv(r#import path)
df_matches = pd.read_csv(r#import path)

#Left join df_deliveries on df_matches on match_id
df = df_deliveries.merge(df_matches, on='match_id', how='left')

/var/folders/20/lyxrxm1j5pv67pzk8kc229lh0000gn/T/ipykernel_39078/3929933423.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df_deliveries = pd.read_csv(r'/Users/LeeT19/Desktop/Net Win Contribution/T20i Data/t20i_deliveries_data.csv')


In [211]:
#Only include matches where teams have full-member ICC status to ensure gameplay is reflective of the top level. 
#Note that matches involving Afghanistan has been removed from the dataset

teams = ['Sri Lanka', 'India', 'South Africa', 'West Indies', 'England', 'New Zealand', 'Australia', 'Ireland', 'Pakistan', 'Zimbabwe', 'Bangladesh']

df = df[df['batting_team'].isin(teams) & df['bowling_team'].isin(teams)]

Game State Features

In [212]:
#Identify the match innings. There are only 2 batting innings (1 per team) in T20 cricket.
first = (
    ((df['toss_decision'] == 'bat')   & (df['batting_team'] == df['toss_winner'])) |
    ((df['toss_decision'] == 'field') & (df['batting_team'] != df['toss_winner']))
)
df['match_innings'] = np.where(first, 1, 2)

In [213]:
#Calculate first innings score and append it to the second innings
first = (df[df['match_innings'] == 1].groupby('match_id')['innings_score'].max())
df['first_innings_score'] = (df['match_id'].map(first).where(df['match_innings'] == 2))

In [214]:
#Identify legal deliverues
df['legal_delivery'] = ((df['wides'] == 0) & (df['noballs'] == 0)).astype(int)

#Create the cumulative count of legal deliveries
df = df.sort_values(['match_id', 'match_innings', 'over', 'ball'])
df['cumulative_innings_legal_balls'] = (df.groupby(['match_id', 'match_innings'])['legal_delivery'].cumsum())

#Create the legal deliveries remaining column
df['legal_deliveries_remaining'] = 120 - df['cumulative_innings_legal_balls']

In [215]:
#1st innings complete criteria
df_innings_1_last = df[df['match_innings'] == 1].groupby('match_id', as_index=False).tail(1)
df_innings_1_complete = df_innings_1_last[
    (df_innings_1_last['innings_wickets'] == 10) | #All 10 wickets taken
    (df_innings_1_last['cumulative_innings_legal_balls'] == 120) #All 120 legal deliveries bowled
]

#2nd innings complete criteria
df_innings_2_last = df[df['match_innings'] == 2].groupby('match_id', as_index=False).tail(1)
df_innings_2_complete = df_innings_2_last[
    (df_innings_2_last['innings_wickets'] == 10) | #All 10 wickets taken
    (df_innings_2_last['cumulative_innings_legal_balls'] == 120) | #All 120 legal deliveries bowled
    (df_innings_2_last['innings_score'] > df_innings_2_last['first_innings_score']) #The 2nd innings batting team passes the first innings score
]

complete_matches = set(df_innings_1_complete['match_id']).intersection(df_innings_2_complete['match_id'])
print(f"Complete matches: {len(complete_matches)}")

df = df[df['match_id'].isin(complete_matches)].reset_index(drop=True)
print(df.shape)


Complete matches: 1601
(379658, 40)


In [216]:
#Identify the powerplay (=1) - the first six overs in each match_innings where only 2 fielders are available ouside the ring
powerplay_overs = [0, 1, 2, 3, 4, 5]
df['powerplay'] = df['over'].isin(powerplay_overs).astype(int)

In [217]:
#Curent run rate - the mean number of runs scored per over in the current innings
df['current_run_rate'] = (df['innings_score'] / df['cumulative_innings_legal_balls']) *6

In [218]:
#Required run rate (second innings only) - the number of runs required per over to win the match
#Make this for match_innings == 2 only match_innings == 1 should be nan
df['required_run_rate'] = (((df['first_innings_score'] + 1) - df['innings_score']) / df['legal_deliveries_remaining']) * 6
df['required_run_rate'] = df['required_run_rate'].replace([np.inf, -np.inf], np.nan)

In [219]:
#Run rate differential (second innings only) - the difference between the required run rate and the current run rate
df['run_rate_differential'] = df['current_run_rate'] - df['required_run_rate']
df['run_rate_differential'] = df['run_rate_differential'].where(df['match_innings'] == 2)

In [220]:
#Runs scored in the previous 10 deliveries
df['runs_scored_prev_10'] = (df.groupby(['match_id', 'match_innings'])['runs_total'].transform(lambda x: x.shift(1).rolling(10, min_periods=1).sum()))

In [221]:
#Wickets taken in previous 10 deliveries
df['is_wicket_binary'] = (df['is_wicket'] == 'TRUE').astype(int)
df['wickets_prev_10'] = (df.groupby(['match_id', 'match_innings'])['is_wicket_binary'].transform(lambda x: x.shift(1).rolling(10, min_periods=1).sum()))

Batting and Bowling Quality Remaining Features

In [222]:
# Individual batters in T20 cricket can be characterised by four standard measures:
#
#   Strike rate      Runs scored per 100 deliveries faced (higher is better).
#                    Captures scoring speed, the binding constraint in a 120-ball innings.
#   Batting average  Runs scored per dismissal (higher is better).
#                    Captures survival; undefined for a batter never dismissed (major limitation of batting avg).
#   Boundary %       Proportion of a batter's runs scored in fours and sixes (higher is better).
#                    Separates batters who score in bursts from those who rotate strike.
#   Dot ball %       Share of deliveries faced returning no run off the bat (lower is better).
#
# The aim of this feature set is to describe the batting resources available to the
# batting side at delivery d: the quality of the two batters at the crease (striker,
# non-striker) and of those still to come. Batting resources deplete as wickets fall,
# so a side four down with recognised batters remaining is in a materially different
# position from one four down with only the tail to follow — a distinction that a
# wickets-remaining count alone cannot make.


In [223]:
# Individual bowlers in T20 cricket can be characterised by four standard measures:
#
#   Economy rate     Runs conceded per over (lower is better).
#                    The primary currency in a format where containment usually
#                    matters more than dismissals.
#   Bowling strike   Deliveries bowled per wicket taken (lower is better).
#   rate             Captures wicket-taking threat independent of runs leaked.
#   Bowling average  Runs conceded per wicket taken (lower is better).
#                    Note this is a deterministic function of the two above:
#                    average = economy x strike rate / 6, so it carries no
#                    information beyond them.
#   Dot ball %       Proportion of deliveries conceding no run (higher is better).
#
# The aim of this feature set is to describe the bowling resources available to the
# fielding side at delivery d: the quality of the bowler in operation and of the
# overs still to be bowled. Unlike batting resources, which deplete through
# dismissals, bowling resources are constrained by the four-over cap per bowler —
# a side may have its best bowler available but no overs left to give them.

Environmental Conditions Features

In [224]:
#Map each venue with their associated country
venue_country = {

    # Antigua and Barbuda
    # NOTE: West Indies venues are grouped below under "West Indies"

    # Australia
    'Adelaide Oval': 'Australia',
    'Allan Border Field': 'Australia',
    'Allan Border Field, Brisbane': 'Australia',
    'Bellerive Oval': 'Australia',
    'Bellerive Oval, Hobart': 'Australia',
    'Brisbane Cricket Ground, Woolloongabba': 'Australia',
    'Brisbane Cricket Ground, Woolloongabba, Brisbane': 'Australia',
    'Carrara Oval': 'Australia',
    "Cazaly's Stadium, Cairns": 'Australia',
    'Great Barrier Reef Arena, Mackay': 'Australia',
    'Junction Oval': 'Australia',
    'Manuka Oval': 'Australia',
    'Manuka Oval, Canberra': 'Australia',
    'Marrara Stadium, Darwin': 'Australia',
    'Melbourne Cricket Ground': 'Australia',
    'North Sydney Oval': 'Australia',
    'North Sydney Oval, Sydney': 'Australia',
    'Perth Stadium': 'Australia',
    'Simonds Stadium, South Geelong': 'Australia',
    'Stadium Australia': 'Australia',
    'Sydney Cricket Ground': 'Australia',
    'Sydney Showground Stadium': 'Australia',
    'W.A.C.A. Ground': 'Australia',
    'Western Australia Cricket Association Ground': 'Australia',

    # Bangladesh
    'Bir Sreshtho Flight Lieutenant Matiur Rahman Stadium, Chattogram': 'Bangladesh',
    'Shere Bangla National Stadium': 'Bangladesh',
    'Shere Bangla National Stadium, Mirpur': 'Bangladesh',
    'Sheikh Abu Naser Stadium': 'Bangladesh',
    'Sylhet International Cricket Stadium': 'Bangladesh',
    'Sylhet International Cricket Stadium, Academy Ground': 'Bangladesh',
    'Sylhet Stadium': 'Bangladesh',
    'Zahur Ahmed Chowdhury Stadium': 'Bangladesh',
    'Zahur Ahmed Chowdhury Stadium, Chattogram': 'Bangladesh',

    # Canada
    'Maple Leaf North-West Ground': 'Canada',

    # China
    'Guanggong International Cricket Stadium': 'China',
    'Zhejiang University of Technology Cricket Field': 'China',

    # Dominica / Grenada / Guyana / etc. grouped as West Indies

    # England
    'Arundel Castle Cricket Club Ground': 'England',
    'County Ground': 'England',
    'County Ground, Bristol': 'England',
    'County Ground, Chelmsford': 'England',
    'County Ground, Derby': 'England',
    'County Ground, Hove': 'England',
    'County Ground, Northampton': 'England',
    'County Ground, New Road, Worcester': 'England',
    'Edgbaston': 'England',
    'Edgbaston, Birmingham': 'England',
    'Headingley, Leeds': 'England',
    'Haslegrave Ground': 'England',
    'Kennington Oval': 'England',
    'Kennington Oval, London': 'England',
    "Lord's": 'England',
    "Lord's, London": 'England',
    'Old Trafford': 'England',
    'Old Trafford, Manchester': 'England',
    'Riverside Ground': 'England',
    'Riverside Ground, Chester-le-Street': 'England',
    'St Lawrence Ground': 'England',
    'St Lawrence Ground, Canterbury': 'England',
    'Sophia Gardens': 'England',
    'Sophia Gardens, Cardiff': 'England',
    'The Cooper Associates County Ground': 'England',
    'The Cooper Associates County Ground, Taunton': 'England',
    'The Rose Bowl': 'England',
    'The Rose Bowl, Southampton': 'England',
    'Trent Bridge': 'England',
    'Trent Bridge, Nottingham': 'England',

    # Ireland
    'Bready': 'Ireland',
    'Bready Cricket Club, Magheramason, Bready': 'Ireland',
    'Castle Avenue, Dublin': 'Ireland',
    'Civil Service Cricket Club, Stormont, Belfast': 'Ireland',
    'Clontarf Cricket Club Ground, Dublin': 'Ireland',
    'Malahide, Dublin': 'Ireland',
    'Pembroke Cricket Club, Sandymount': 'Ireland',
    'Pembroke Cricket Club, Sandymount, Dublin': 'Ireland',
    'Sydney Parade': 'Ireland',
    'The Village, Malahide': 'Ireland',
    'The Village, Malahide, Dublin': 'Ireland',

    # India
    'Arun Jaitley Stadium': 'India',
    'Arun Jaitley Stadium, Delhi': 'India',
    'Barabati Stadium': 'India',
    'Barabati Stadium, Cuttack': 'India',
    'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium': 'India',
    'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow': 'India',
    'Brabourne Stadium': 'India',
    'Brabourne Stadium, Mumbai': 'India',
    'Dr DY Patil Sports Academy, Mumbai': 'India',
    'Dr DY Patil Sports Academy, Navi Mumbai': 'India',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium': 'India',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam': 'India',
    'Eden Gardens': 'India',
    'Eden Gardens, Kolkata': 'India',
    'Feroz Shah Kotla': 'India',
    'Green Park': 'India',
    'Himachal Pradesh Cricket Association Stadium': 'India',
    'Himachal Pradesh Cricket Association Stadium, Dharamsala': 'India',
    'Holkar Cricket Stadium': 'India',
    'Holkar Cricket Stadium, Indore': 'India',
    'JSCA International Stadium Complex': 'India',
    'JSCA International Stadium Complex, Ranchi': 'India',
    'Lalabhai Contractor Stadium': 'India',
    'M Chinnaswamy Stadium': 'India',
    'M Chinnaswamy Stadium, Bengaluru': 'India',
    'M.Chinnaswamy Stadium': 'India',
    'MA Chidambaram Stadium, Chepauk': 'India',
    'MA Chidambaram Stadium, Chepauk, Chennai': 'India',
    'Maharaja Yadavindra Singh International Cricket Stadium, New Chandigarh': 'India',
    'Maharashtra Cricket Association Stadium': 'India',
    'Maharashtra Cricket Association Stadium, Pune': 'India',
    'Narendra Modi Stadium': 'India',
    'Narendra Modi Stadium, Ahmedabad': 'India',
    'Punjab Cricket Association IS Bindra Stadium, Mohali': 'India',
    'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh': 'India',
    'Punjab Cricket Association Stadium, Mohali': 'India',
    'Rajiv Gandhi International Stadium, Uppal': 'India',
    'Rajiv Gandhi International Stadium, Uppal, Hyderabad': 'India',
    'Sardar Patel Stadium, Motera': 'India',
    'Saurashtra Cricket Association Stadium': 'India',
    'Saurashtra Cricket Association Stadium, Rajkot': 'India',
    'Sawai Mansingh Stadium, Jaipur': 'India',
    'Shaheed Veer Narayan Singh International Stadium, Raipur': 'India',
    'Shrimant Madhavrao Scindia Cricket Stadium, Gwalior': 'India',
    'Subrata Roy Sahara Stadium': 'India',
    'Vidarbha Cricket Association Stadium, Jamtha': 'India',
    'Vidarbha Cricket Association Stadium, Jamtha, Nagpur': 'India',
    'Wankhede Stadium': 'India',
    'Wankhede Stadium, Mumbai': 'India',
    'Barsapara Cricket Stadium': 'India',
    'Barsapara Cricket Stadium, Guwahati': 'India',
    'Greenfield International Stadium': 'India',
    'Greenfield International Stadium, Thiruvananthapuram': 'India',

    # Kenya
    'Gymkhana Club Ground': 'Kenya',

    # Malaysia
    'Kinrara Academy Oval': 'Malaysia',
    'Kinrara Academy Oval, Kuala Lumpur': 'Malaysia',
    'Royal Selangor Club': 'Malaysia',

    # Nepal
    'Mulpani Cricket Ground': 'Nepal',

    # Netherlands
    'Sportpark Maarschalkerweerd': 'Netherlands',

    # New Zealand
    'AMI Stadium': 'New Zealand',
    'Basin Reserve': 'New Zealand',
    'Basin Reserve, Wellington': 'New Zealand',
    'Bay Oval': 'New Zealand',
    'Bay Oval, Mount Maunganui': 'New Zealand',
    'Eden Park': 'New Zealand',
    'Eden Park, Auckland': 'New Zealand',
    'Hagley Oval': 'New Zealand',
    'Hagley Oval, Christchurch': 'New Zealand',
    'Jade Stadium': 'New Zealand',
    'John Davies Oval, Queenstown': 'New Zealand',
    'McLean Park': 'New Zealand',
    'McLean Park, Napier': 'New Zealand',
    'Pukekura Park': 'New Zealand',
    'Saxton Oval': 'New Zealand',
    'Saxton Oval, Nelson': 'New Zealand',
    'Seddon Park': 'New Zealand',
    'Seddon Park, Hamilton': 'New Zealand',
    'Sky Stadium': 'New Zealand',
    'Sky Stadium, Wellington': 'New Zealand',
    'University Oval': 'New Zealand',
    'University Oval, Dunedin': 'New Zealand',
    'Westpac Stadium': 'New Zealand',

    # Pakistan
    'Gaddafi Stadium': 'Pakistan',
    'Gaddafi Stadium, Lahore': 'Pakistan',
    'Multan Cricket Stadium': 'Pakistan',
    'National Stadium': 'Pakistan',
    'National Stadium, Karachi': 'Pakistan',
    'Rawalpindi Cricket Stadium': 'Pakistan',
    'Southend Club Cricket Stadium': 'Pakistan',
    'Southend Club Cricket Stadium, Karachi': 'Pakistan',

    # Qatar
    'West End Park International Cricket Stadium': 'Qatar',

    # Scotland
    'Forthill': 'Scotland',

    # South Africa
    'Boland Bank Park': 'South Africa',
    'Boland Park': 'South Africa',
    'Boland Park, Paarl': 'South Africa',
    'Buffalo Park': 'South Africa',
    'Buffalo Park, East London': 'South Africa',
    'City Oval': 'South Africa',
    'De Beers Diamond Oval': 'South Africa',
    'Diamond Oval, Kimberley': 'South Africa',
    'Kerrydale Oval': 'South Africa',
    'Kingsmead': 'South Africa',
    'Kingsmead, Durban': 'South Africa',
    'LC de Villiers Oval': 'South Africa',
    'Mangaung Oval': 'South Africa',
    'Moses Mabhida Stadium': 'South Africa',
    'New Wanderers Stadium': 'South Africa',
    'Newlands': 'South Africa',
    'Newlands, Cape Town': 'South Africa',
    'OUTsurance Oval': 'South Africa',
    'Senwes Park': 'South Africa',
    'Senwes Park, Potchefstroom': 'South Africa',
    "St George's Park": 'South Africa',
    "St George's Park, Gqeberha": 'South Africa',
    'SuperSport Park': 'South Africa',
    'SuperSport Park, Centurion': 'South Africa',
    'The Wanderers Stadium': 'South Africa',
    'The Wanderers Stadium, Johannesburg': 'South Africa',
    'Willowmoore Park': 'South Africa',
    'Willowmoore Park, Benoni': 'South Africa',
    'YMCA Cricket Club': 'South Africa',

    # Sri Lanka
    'Chilaw Marians Cricket Club Ground': 'Sri Lanka',
    'Colombo Cricket Club Ground': 'Sri Lanka',
    'Colts Cricket Club Ground': 'Sri Lanka',
    'Galle International Stadium': 'Sri Lanka',
    'Mahinda Rajapaksa International Cricket Stadium, Sooriyawewa': 'Sri Lanka',
    'Mahinda Rajapaksa International Cricket Stadium, Sooriyawewa, Hambantota': 'Sri Lanka',
    'Mercantile Cricket Association Ground': 'Sri Lanka',
    'Nondescripts Cricket Club Ground': 'Sri Lanka',
    'P Sara Oval': 'Sri Lanka',
    'P Sara Oval, Colombo': 'Sri Lanka',
    'Pallekele International Cricket Stadium': 'Sri Lanka',
    'R Premadasa Stadium': 'Sri Lanka',
    'R Premadasa Stadium, Colombo': 'Sri Lanka',
    'R.Premadasa Stadium, Khettarama': 'Sri Lanka',
    'Rangiri Dambulla International Stadium': 'Sri Lanka',
    'Sinhalese Sports Club Ground': 'Sri Lanka',
    'Sinhalese Sports Club Ground, Colombo': 'Sri Lanka',

    # Thailand
    'Asian Institute of Technology Ground': 'Thailand',
    'Terdthai Cricket Ground': 'Thailand',

    # United Arab Emirates
    'Dubai International Cricket Stadium': 'United Arab Emirates',
    'Sharjah Cricket Stadium': 'United Arab Emirates',
    'Sheikh Zayed Stadium': 'United Arab Emirates',
    'Sheikh Zayed Stadium, Abu Dhabi': 'United Arab Emirates',
    'Zayed Cricket Stadium, Abu Dhabi': 'United Arab Emirates',

    # United States
    'Central Broward Regional Park Stadium Turf Ground': 'United States',
    'Central Broward Regional Park Stadium Turf Ground, Lauderhill': 'United States',
    'Grand Prairie Stadium, Dallas': 'United States',
    'Nassau County International Cricket Stadium, New York': 'United States',

    # West Indies
    'Arnos Vale Ground, Kingstown': 'West Indies',
    'Arnos Vale Ground, Kingstown, St Vincent': 'West Indies',
    'Beausejour Stadium, Gros Islet': 'West Indies',
    'Brian Lara Stadium, Tarouba': 'West Indies',
    'Brian Lara Stadium, Tarouba, Trinidad': 'West Indies',
    'Coolidge Cricket Ground': 'West Indies',
    'Coolidge Cricket Ground, Antigua': 'West Indies',
    'Daren Sammy National Cricket Stadium, Gros Islet': 'West Indies',
    'Daren Sammy National Cricket Stadium, Gros Islet, St Lucia': 'West Indies',
    'Darren Sammy National Cricket Stadium, St Lucia': 'West Indies',
    'Kensington Oval, Barbados': 'West Indies',
    'Kensington Oval, Bridgetown': 'West Indies',
    'Kensington Oval, Bridgetown, Barbados': 'West Indies',
    'National Cricket Stadium, Grenada': 'West Indies',
    "National Cricket Stadium, St George's": 'West Indies',
    "National Cricket Stadium, St George's, Grenada": 'West Indies',
    'Providence Stadium': 'West Indies',
    'Providence Stadium, Guyana': 'West Indies',
    "Queen's Park Oval, Port of Spain": 'West Indies',
    'Sabina Park, Kingston': 'West Indies',
    'Sabina Park, Kingston, Jamaica': 'West Indies',
    'Sir Vivian Richards Stadium, North Sound': 'West Indies',
    'Sir Vivian Richards Stadium, North Sound, Antigua': 'West Indies',
    'Three Ws Oval, Cave Hill, Barbados': 'West Indies',
    'Warner Park, Basseterre': 'West Indies',
    'Warner Park, Basseterre, St Kitts': 'West Indies',
    'Warner Park, St Kitts': 'West Indies',
    'Windsor Park, Roseau': 'West Indies',
    'Windsor Park, Roseau, Dominica': 'West Indies',

    # Zimbabwe
    'Harare Sports Club': 'Zimbabwe',
    'Queens Sports Club': 'Zimbabwe',
    'Queens Sports Club, Bulawayo': 'Zimbabwe'
}

df['venue_country'] = df['venue'].map(venue_country)

#Identify any venue_country not mapped
print(df['venue_country'].isna().sum())
print(df.loc[df['venue_country'].isna(), 'venue'].value_counts())

0
Series([], Name: count, dtype: int64)


In [225]:
#Mark 'batter_home_ground' as 1 if batting_team == venue_country
df['batter_home_ground'] = (df['venue_country'] == df['batting_team']).astype(int)

In [226]:
#Create the target variable 'batting_team_win'
df['batting_team_win_match'] = (df['match_winner'] == df['batting_team']).astype(int)
print(df.columns)

Index(['match_id', 'date_x', 'venue', 'batting_team', 'bowling_team', 'over',
       'ball', 'striker', 'bowler', 'non_striker', 'runs_batter',
       'runs_extras', 'runs_total', 'innings_score', 'innings_wickets',
       'is_wicket', 'player_out', 'wicket_kind', 'match_winner',
       'winning_outcome', 'toss_winner', 'toss_decision', 'match_referee',
       'tv_umpire', 'reserve_umpire', 'umpire_1', 'umpire_2', 'wides',
       'noballs', 'byes', 'legbyes', 'penalty', 'date_y', 'gender', 'game',
       'match_innings', 'first_innings_score', 'legal_delivery',
       'cumulative_innings_legal_balls', 'legal_deliveries_remaining',
       'powerplay', 'current_run_rate', 'required_run_rate',
       'run_rate_differential', 'runs_scored_prev_10', 'is_wicket_binary',
       'wickets_prev_10', 'venue_country', 'batter_home_ground',
       'batting_team_win_match'],
      dtype='object')


In [227]:
#Export dataset with columns of interest for downstream functions
cols = [
    'date_x',
    'venue',
    'batting_team',
    'bowling_team',
    'over',
    'ball',
    'striker',
    'bowler',
    'non_striker',
    'runs_batter',
    'runs_extras',
    'runs_total',
    'innings_score',
    'innings_wickets',
    'is_wicket_binary',
    'player_out',
    'wicket_kind',
    'winning_outcome',
    'toss_winner',
    'toss_decision',
    'wides',
    'noballs',
    'byes',
    'legbyes',
    'penalty',
    'gender', #Ensure to segregate model training by gender
    'match_innings',
    'first_innings_score',
    'legal_delivery',
    'cumulative_innings_legal_balls',
    'legal_deliveries_remaining',
    'powerplay',
    'current_run_rate',
    'required_run_rate',
    'run_rate_differential',
    'runs_scored_prev_10',
    'wickets_prev_10',
    'venue_country',
    'batter_home_ground',
    'batting_team_win_match',
]

df = df[cols]
#Export your dataset